In [0]:
from datetime import datetime
 
CONFIG = {
    "catalog":      "workspace",
    "schema":       "banking_datasentry",
    "volume_path":  "/Volumes/workspace/banking_datasentry/datasentry_files",
}
 
TABLE_CONFIG = [
    {"pattern": "customers",    "table": "bronze_customers",    "required": True},
    {"pattern": "accounts",     "table": "bronze_accounts",     "required": True},
    {"pattern": "transactions", "table": "bronze_transactions",  "required": True},
]
 
# Timestamp for audit column — same value used across all tables in this run
INGESTED_AT = datetime.now().isoformat()
 
print(f"Catalog  : {CONFIG['catalog']}")
print(f"Schema   : {CONFIG['schema']}")
print(f"Volume   : {CONFIG['volume_path']}")
print(f"Run time : {INGESTED_AT}")

In [0]:
spark.sql(f"USE CATALOG {CONFIG['catalog']}")
spark.sql(f"USE SCHEMA {CONFIG['schema']}")
 
print(f"Default catalog and schema set to: {CONFIG['catalog']}.{CONFIG['schema']}")

In [0]:
def discover_files(volume_path: str, pattern: str) -> list:
    """
    List all files in volume_path whose name starts with `pattern`.
    Returns a list of full file paths.
    """
    try:
        all_files = dbutils.fs.ls(volume_path)
    except Exception as e:
        print(f"  ERROR: Could not list Volume path {volume_path}: {e}")
        return []
 
    matched = [
        f.path for f in all_files
        if f.name.startswith(pattern) and f.name.endswith(".csv")
    ]
    return matched

In [0]:
from pyspark.sql import functions as F
 
def ingest_to_bronze(file_path: str, table_name: str, ingested_at: str):
    """
    Read a CSV file and append it to a Bronze Delta table.
    All columns are read as strings. Two audit columns are added.
    """
    print(f"  Reading : {file_path}")
 
    # Read CSV — header=True uses first row as column names
    # inferSchema=False keeps everything as string (safe for Bronze)
    df = spark.read.option("header", True) \
                   .option("inferSchema", False) \
                   .csv(file_path)
 
    row_count = df.count()
    print(f"  Rows read : {row_count:,}")
 
    # Add audit columns
    df = df.withColumn("ingested_at",  F.lit(ingested_at)) \
           .withColumn("source_file",  F.lit(file_path.split("/")[-1]))
 
    # Write to Delta — APPEND so we never overwrite raw history
    df.write.format("delta") \
            .mode("append") \
            .saveAsTable(f"{table_name}")
 
    print(f"  Written  : {table_name} (+{row_count:,} rows)")
    return row_count

In [0]:
print("=" * 60)
print("BRONZE INGESTION — START")
print("=" * 60)
 
ingestion_summary = []
 
for config in TABLE_CONFIG:
    pattern   = config["pattern"]
    table     = config["table"]
    required  = config["required"]
 
    print(f"\n[{pattern.upper()}]")
 
    # Step 1: Discover matching files in Volume
    matched_files = discover_files(CONFIG["volume_path"], pattern)
 
    if not matched_files:
        status = "ERROR" if required else "SKIPPED"
        msg = f"No files found matching pattern '{pattern}*.csv' in Volume"
        print(f"  {status}: {msg}")
        ingestion_summary.append({
            "table": table, "files_found": 0,
            "rows_ingested": 0, "status": status
        })
        continue
 
    print(f"  Files found: {len(matched_files)}")
 
    # Step 2: Ingest each matched file
    total_rows = 0
    for fpath in matched_files:
        try:
            rows = ingest_to_bronze(fpath, table, INGESTED_AT)
            total_rows += rows
        except Exception as e:
            print(f"  ERROR ingesting {fpath}: {e}")
            ingestion_summary.append({
                "table": table, "files_found": len(matched_files),
                "rows_ingested": 0, "status": "FAILED"
            })
            continue
 
    ingestion_summary.append({
        "table": table, "files_found": len(matched_files),
        "rows_ingested": total_rows, "status": "SUCCESS"
    })
 
print("\n" + "=" * 60)
print("BRONZE INGESTION — COMPLETE")
print("=" * 60)

In [0]:
print(f"\n{'TABLE':<30} {'FILES':>6} {'ROWS':>12} {'STATUS':<10}")
print("-" * 62)
for row in ingestion_summary:
    print(f"{row['table']:<30} {row['files_found']:>6} {row['rows_ingested']:>12,} {row['status']:<10}")

In [0]:
print("\n" + "=" * 60)
print("VERIFICATION — READING BRONZE TABLES BACK")
print("=" * 60)
 
for config in TABLE_CONFIG:
    table = config["table"]
    try:
        df = spark.read.format("delta").table(table)
        print(f"\n[{table.upper()}]")
        print(f"  Row count : {df.count():,}")
        print(f"  Columns   : {len(df.columns)}")
        print(f"  Schema (first 5 cols):")
        for field in df.schema.fields[:5]:
            print(f"    {field.name} : {field.dataType}")
        print(f"  Sample row:")
        df.limit(1).show(truncate=50, vertical=True)
    except Exception as e:
        print(f"  Could not read {table}: {e}")
 
print("\nNotebook 02 complete. Bronze Delta tables ready for Silver validation.")